In [ ]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake
from firedrake import *
import matplotlib.pyplot as plt
import numpy as np

# Ex.1 - Stokes: GMRES and optimal preconditioning

We consider the following cavity problem

\begin{equation*}
\begin{cases}
- \Delta \boldsymbol{u} + \nabla  p  = \boldsymbol{0} & {\rm in} \ \Omega=(0,1)^2, \\
{\rm div}\,\boldsymbol{u} = 0 & {\rm in} \ \Omega, \\
\boldsymbol{u} = \boldsymbol{g}_\text{D} & {\rm on} \ \Gamma_4.\\
\boldsymbol{u} = \boldsymbol{0} & {\rm on} \ \partial\Omega\setminus\Gamma_4, \\
\end{cases}
\end{equation*}

with $\boldsymbol{g}_\text{D} = 1\boldsymbol{i}$. We solve this problem using GMRES and the optimal diagonal preconditioner

$$
P = \begin{bmatrix}
A & 0 \\
0 & \frac{1}{\nu}M_p
\end{bmatrix},
$$

with $\nu = 1$.

In [ ]:
# Build the mesh

N = 20
mesh = UnitSquareMesh(N, N)

fig, ax = plt.subplots()
triplot(mesh, axes=ax)
ax.legend()

In [ ]:
# Stokes problem

# Define function space and trial/test functions for velocity and pressure
V = VectorFunctionSpace(mesh, "P", 2)
Q = FunctionSpace(mesh, "P", 1)
W = MixedFunctionSpace([V, Q])
u, p = TrialFunctions(W)
v, q = TestFunctions(W)
print('Number of DOF \n - u    :',V.dim(),' \n - p    :',Q.dim(),' \n - total:',W.dim(), "\n")

# Other choices of function spaces:

# P1-P0
# V = VectorFunctionSpace(mesh, 'P', 1)
# Q = FunctionSpace(mesh, 'DP', 0) # NB: P0 are DISCONTINUOUS elements (DP)

# # P1-P1
# V = VectorFunctionSpace(mesh, 'P', 1)
# Q = FunctionSpace(mesh, 'P', 1)

# # P1b-P1
# # The enrichment of the velocity space has to be done at the finite element level
# V1_el = FiniteElement('CG', mesh.ufl_cell(), 1)
# B_el = FiniteElement('Bubble', mesh.ufl_cell(), mesh.topological_dimension + 1)
# V_el = VectorElement(NodalEnrichedElement(V1_el, B_el))
# V = FunctionSpace(mesh, V_el)
# Q = FunctionSpace(mesh, 'P', 1)

# Define boundary conditions for velocity (Dirichlet BCs)
bcs = [DirichletBC(W.sub(0), Constant((1.0, 0.0)), 4), # Top boundary
       DirichletBC(W.sub(0), Constant((0.0, 0.0)), (1,2,3)) # Bottom, left, right boundaries
       ]

# Define the variational problem
a = inner(grad(u), grad(v)) * dx - p * div(v) * dx + q * div(u) * dx
L = inner(Constant((0.0, 0.0)), v) * dx

# !!!!!!!!!!!
# Solve the problem passing the null space for fully Dirichlet conditions 
w_h = Function(W)
parameters = {'ksp_type': 'gmres', 'pc_type': 'ilu', 'ksp_rtol': 1.e-8, 'ksp_max_it': 10000}
nullspace = MixedVectorSpaceBasis(W, [W.sub(0), VectorSpaceBasis(constant=True, comm=COMM_WORLD)])
variation_problem = LinearVariationalProblem(a, L, w_h, bcs=bcs)
solver = LinearVariationalSolver(variation_problem, solver_parameters=parameters, nullspace=nullspace)
solver.solve()
# Print the number of iterations and the final residual norm
print("Iterations with GMRES (tol = 1e-8):", solver.snes.ksp.getIterationNumber())
print("Final residual norm:", solver.snes.ksp.getResidualNorm())

In [ ]:
# Plot velocity and pressure
u_h, p_h = w_h.subfunctions

fig, ax = plt.subplots()
q = quiver(u_h, axes=ax)
fig.colorbar(q)
ax.set_title("Velocity field")

fig, ax = plt.subplots()
q = tripcolor(p_h, axes=ax)
fig.colorbar(q)
ax.set_title("Pressure field")